In [12]:
import pandas as pd

In [10]:
from langchain_core.messages import HumanMessage
prompt = "test"
type([HumanMessage(prompt)][-1])

langchain_core.messages.human.HumanMessage

In [13]:
# basic imformation
year = 1996
month = 11
day = 6
hour = 16  # 子時
is_male = True
is_lunar = False

In [14]:
from ziweidoushu.kernel import ZiweiChart
from ziweidoushu.base import ZiWeiConfig

ziwei_inst = ZiweiChart(ZiWeiConfig(year, month, day, hour, is_male, is_lunar))
df = ziwei_inst.gen_chart()

干支: 丙子 年, 生於 申 時
轉換成農曆生日： 1996年 9月 26日
命宮:  寅 身宮:  午
五行局:  木三局
紫薇宮:  酉


In [15]:
ziwei_domain_df = pd.read_csv(("basic_document/star_palace_meaning.csv"))

In [16]:
from sentence_transformers import SentenceTransformer, util

# 初始化 BERT 模型
model = SentenceTransformer('joe32140/ModernBERT-large-msmarco')


c:\Users\jerry\anaconda3\envs\ziwei-agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:

# 對所有 object 建立向量
object_embeddings = model.encode(ziwei_domain_df['object'].tolist(), convert_to_tensor=True)

Compiling the model with `torch.compile` and using a `torch.cpu` device is not supported. Falling back to non-compiled mode.


In [18]:
object_embeddings

tensor([[-1.8865, -0.6000, -0.8712,  ...,  0.1908,  0.6233,  0.6911],
        [ 0.6750, -0.3377,  0.6473,  ...,  0.2885,  0.0582, -1.1102],
        [ 0.5486, -0.2450, -0.4586,  ...,  0.8868, -1.4301, -0.0323],
        ...,
        [ 0.3141,  1.6709, -0.1375,  ..., -0.2216, -1.1923,  1.1808],
        [ 0.4562,  0.3246,  0.0567,  ...,  0.1209, -0.6946,  0.4807],
        [ 0.4765, -0.8393,  0.3168,  ...,  0.4569, -0.8297, -0.5670]])

In [7]:
from chat_bot.utils import build_llm, process_llm_output

from langchain.prompts import PromptTemplate
from chat_bot.prompt import KEYWORD_PROMPT, ZIWEI_PROMPT, REPORT_PROMPT

In [33]:
def split_keywords(context:str, model:str = "gpt-3.5") -> list:
    llm = build_llm(model)
    keyword_prompt = PromptTemplate.from_template(KEYWORD_PROMPT)
    chain = keyword_prompt | llm
    return process_llm_output(chain.invoke({"sentence": context}).content).split(",")

def retrieve_definitions(df:pd.DataFrame, keywords:list, top_k=1) -> str:
    """
    根據輸入的關鍵詞列表，回傳每個詞對應的最佳定義
    """
    results = {"object":[], "definition":[]}
    for keyword in keywords:
        keyword_embedding = model.encode(keyword, convert_to_tensor=True)
        hits = util.semantic_search(keyword_embedding, object_embeddings, top_k=top_k)[0]
        for hit in hits:
            if hit['score'] < 0.7:
                continue
            # print("keyword: ", keyword)
            # print("hit['score']: ", hit['score'])
            results["object"].append(df.iloc[hit['corpus_id']]['object'])
            results["definition"].append(df.iloc[hit['corpus_id']]['definition'])
    return pd.DataFrame(results).to_markdown()

def analysis_ziwei(context:str, rag_document:str, model:str = "gpt-3.5") -> str:
    llm = build_llm(model)
    ziwei_prompt = PromptTemplate.from_template(ZIWEI_PROMPT)
    chain = ziwei_prompt | llm
    return process_llm_output(chain.invoke({"sentence": context, "rag_report": rag_document}).content)

def build_ziwei_report(ziwei_summary:str, model:str = "gpt-3.5") -> str:
    llm = build_llm(model)
    report_prompt = PromptTemplate.from_template(REPORT_PROMPT)
    chain = report_prompt | llm
    return chain.invoke({"ziwei_summary": ziwei_summary}).content


def format_row(row):
    # 處理星
    if row["星"]:
        stars = [f"{star.strip()}星" for star in row["星"].split(",")]
        star_str = ", ".join(stars)
        # 組成句子
        sentence = f"{star_str}入{row['宮']}位於{row['天干']}{row['地支']}"
    else:
        sentence = f"沒有星落入{row['宮']}位於{row['天干']}{row['地支']}"
    # 加入化
    if row["化"]:
        sentence += f"，化{row['化']}"
    else:
        sentence += "，沒有化星"
    # 加入位置
    return sentence

def get_palace_information(df:pd.DataFrame):
    result_string = ""
    for _, row in df.iterrows():
        sentence = format_row(row)
        keyword_list = split_keywords(sentence)
        rag_document = retrieve_definitions(ziwei_domain_df, keyword_list)
        result_string += analysis_ziwei(sentence, rag_document)
    return result_string


In [35]:
ziwei_report = get_palace_information(df)
print(ziwei_report)

keyword:  天梁星
hit['score']:  0.856757640838623
keyword:   文曲星
hit['score']:  0.7792748212814331
keyword:   左輔星
hit['score']:  0.7645536661148071
keyword:   夫妻宮
hit['score']:  1.0
keyword:  七殺星
hit['score']:  0.7360419034957886
keyword:   廉貞星
hit['score']:  0.7867332696914673
keyword:   兄弟宮
hit['score']:  1.0
keyword:  文昌星
hit['score']:  0.8778478503227234
keyword:  右弼星
hit['score']:  0.7560081481933594
keyword:  命宮
hit['score']:  1.0
keyword:  地空星
hit['score']:  0.7793279886245728
keyword:  陀羅星
hit['score']:  0.7795747518539429
keyword:   天同星
hit['score']:  0.7899402379989624
keyword:   福德宮
hit['score']:  1.0
keyword:  武曲星
hit['score']:  0.7298455238342285
keyword:  破軍星
hit['score']:  0.7965549230575562
keyword:  太陽星
hit['score']:  0.8410129547119141
keyword:  擎羊星
hit['score']:  0.7619575262069702
keyword:  鈴星星
hit['score']:  0.9377681612968445
keyword:  官祿宮
hit['score']:  0.9999999403953552
keyword:  地劫星
hit['score']:  0.711028516292572
keyword:   天府星
hit['score']:  0.775135338306427


In [19]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

ZIWEI_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are a Zi Wei Dou Shu assistant."),
    ("user", "{rag_report}\n\nQuestion: {question}")
])

chain = ZIWEI_PROMPT | llm
output = chain.invoke({"question": question, "rag_report": rag_string})

NameError: name 'llm' is not defined

In [ ]:
d